In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

TRAINING_INPUT = Path(
    "/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-soft-supervised-gpu-training"
)

FINAL_MODEL_PATH = (
    TRAINING_INPUT
    / "soft_supervised_gpu"
    / "final_full_data_model"
    / "final_model_epoch_3.pt"
)

COMP_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

TEST_CSV = COMP_ROOT / "test.csv"
TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
TEST_SERIES_ROOT = COMP_ROOT / "test_series"
SAMPLE_SUB_PATH = COMP_ROOT / "sample_submission.csv"

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# Fallback values if a hidden study cannot be processed at all.
# These roughly follow our training prevalence / model base rates,
# but are only used as an emergency fallback.
FALLBACK = {
    "ACL": 0.18,
    "MCL": 0.11,
    "Medial Meniscus": 0.40,
    "Lateral Meniscus": 0.22,
    "Medial OA": 0.33,
    "Lateral OA": 0.24,
    "PF OA": 0.42,
    "Effusion": 0.47,
    "Synovitis": 0.37,
    "Baker's": 0.28,
    "Contusion": 0.12,
    "Fracture": 0.09,
}

if not FINAL_MODEL_PATH.exists():
    raise FileNotFoundError("Final model checkpoint missing")

test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Studies:", len(test_df))
print("Series:", len(test_series_df))
print("Submission template:", sample_sub.shape)
print("CUDA:", torch.cuda.is_available())

Studies: 3
Series: 15
Submission template: (3, 13)
CUDA: True


In [2]:
import torch.nn as nn
from torchvision.models import resnet18

class KneeMRIClassifier(nn.Module):
    def __init__(self, num_targets=12, dropout=0.20):
        super().__init__()

        self.encoder = resnet18(weights=None)
        self.encoder.fc = nn.Identity()

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(512, num_targets)

    def forward(self, x):
        x = self.encoder(x)
        x = self.dropout(x)
        return self.classifier(x)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = KneeMRIClassifier(
    num_targets=12,
    dropout=0.20
)

checkpoint = torch.load(
    FINAL_MODEL_PATH,
    map_location="cpu"
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]

elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]

else:
    state_dict = checkpoint

model.load_state_dict(
    state_dict,
    strict=True
)

model = model.to(device)
model.eval()

print("Model loaded")

Model loaded


In [3]:
import pydicom
from tqdm.auto import tqdm

def get_slice_position(ds):

    if hasattr(ds, "ImagePositionPatient"):
        try:
            ipp = np.asarray(
                ds.ImagePositionPatient,
                dtype=float
            )

            if len(ipp) == 3:
                return float(ipp[2])

        except Exception:
            pass

    if hasattr(ds, "SliceLocation"):
        try:
            return float(ds.SliceLocation)
        except Exception:
            pass

    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber)
        except Exception:
            pass

    return np.nan


slice_rows = []
bad_dicoms = 0
missing_series = 0

for row in tqdm(
    test_series_df.itertuples(index=False),
    total=len(test_series_df),
    desc="Indexing MRI"
):

    study_uid = str(row.StudyInstanceUID)
    series_uid = str(row.SeriesInstanceUID)

    series_path = (
        TEST_SERIES_ROOT
        / study_uid
        / series_uid
    )

    if not series_path.exists():
        missing_series += 1
        continue

    dcm_files = list(
        series_path.glob("*.dcm")
    )

    for dcm_path in dcm_files:

        try:
            ds = pydicom.dcmread(
                dcm_path,
                stop_before_pixels=True,
                force=True
            )

            position = get_slice_position(ds)

            instance_number = float(
                getattr(
                    ds,
                    "InstanceNumber",
                    np.nan
                )
            )

            spacing_y = np.nan
            spacing_x = np.nan

            try:
                if hasattr(ds, "PixelSpacing"):
                    spacing_y = float(ds.PixelSpacing[0])
                    spacing_x = float(ds.PixelSpacing[1])
            except Exception:
                pass

            slice_rows.append({
                "StudyInstanceUID": study_uid,
                "SeriesInstanceUID": series_uid,
                "Anatomical_Plane":
                    getattr(row, "Anatomical_Plane", "Unknown"),
                "Path": str(dcm_path),
                "Position": position,
                "InstanceNumber": instance_number,
                "PixelSpacingY": spacing_y,
                "PixelSpacingX": spacing_x,
            })

        except Exception:
            bad_dicoms += 1


test_slice_df = pd.DataFrame(slice_rows)

sorted_groups = []

if len(test_slice_df) > 0:

    for _, group in test_slice_df.groupby(
        ["StudyInstanceUID", "SeriesInstanceUID"],
        sort=False
    ):

        group = group.copy()

        # Prefer physical position only when useful.
        if (
            group["Position"].notna().all()
            and group["Position"].nunique() > 1
        ):

            group = group.sort_values(
                ["Position", "InstanceNumber"],
                kind="mergesort"
            )

        else:

            # Safer fallback for sagittal/coronal cases
            group = group.sort_values(
                "InstanceNumber",
                kind="mergesort"
            )

        sorted_groups.append(
            group.reset_index(drop=True)
        )


if sorted_groups:

    test_slice_df = pd.concat(
        sorted_groups,
        ignore_index=True
    )


print("Indexed slices:", len(test_slice_df))
print("Bad DICOMs skipped:", bad_dicoms)
print("Missing series skipped:", missing_series)

Indexing MRI:   0%|          | 0/15 [00:00<?, ?it/s]

Indexed slices: 557
Bad DICOMs skipped: 0
Missing series skipped: 0


In [4]:
triplet_rows = []

if len(test_slice_df) > 0:

    grouped = test_slice_df.groupby(
        ["StudyInstanceUID", "SeriesInstanceUID"],
        sort=False
    )

    for (
        study_uid,
        series_uid
    ), group in tqdm(
        grouped,
        desc="Building triplets"
    ):

        group = group.reset_index(drop=True)

        if len(group) < 3:
            continue

        # Use median valid spacing for robustness
        sy = pd.to_numeric(
            group["PixelSpacingY"],
            errors="coerce"
        )

        sx = pd.to_numeric(
            group["PixelSpacingX"],
            errors="coerce"
        )

        spacing_y = float(
            sy.dropna().median()
        ) if sy.notna().any() else 0.5

        spacing_x = float(
            sx.dropna().median()
        ) if sx.notna().any() else 0.5

        if not np.isfinite(spacing_y) or spacing_y <= 0:
            spacing_y = 0.5

        if not np.isfinite(spacing_x) or spacing_x <= 0:
            spacing_x = 0.5

        for i in range(
            1,
            len(group) - 1
        ):

            triplet_rows.append({
                "StudyInstanceUID": study_uid,
                "SeriesInstanceUID": series_uid,
                "PreviousPath":
                    group.iloc[i - 1]["Path"],
                "CentrePath":
                    group.iloc[i]["Path"],
                "NextPath":
                    group.iloc[i + 1]["Path"],
                "PixelSpacingY": spacing_y,
                "PixelSpacingX": spacing_x,
            })


test_triplets_df = pd.DataFrame(
    triplet_rows
)

print("Triplets:", len(test_triplets_df))

if len(test_triplets_df) > 0:
    print(
        "Covered studies:",
        test_triplets_df[
            "StudyInstanceUID"
        ].nunique()
    )

Building triplets:   0%|          | 0/15 [00:00<?, ?it/s]

Triplets: 527
Covered studies: 3


In [5]:
def read_rescaled_pixels(path):

    ds = pydicom.dcmread(
        path,
        force=True
    )

    arr = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(
            ds,
            "RescaleSlope",
            1.0
        )
    )

    intercept = float(
        getattr(
            ds,
            "RescaleIntercept",
            0.0
        )
    )

    return arr * slope + intercept


percentile_rows = []
bad_percentile_series = 0

if len(test_slice_df) > 0:

    for (
        study_uid,
        series_uid
    ), group in tqdm(
        test_slice_df.groupby(
            ["StudyInstanceUID", "SeriesInstanceUID"],
            sort=False
        ),
        desc="Computing P1/P99"
    ):

        try:

            sampled = []

            for path in group["Path"]:

                try:
                    arr = read_rescaled_pixels(
                        path
                    )
                except Exception:
                    continue

                flat = arr.reshape(-1)

                flat = flat[
                    np.isfinite(flat)
                ]

                if len(flat) == 0:
                    continue

                if len(flat) > 50000:

                    idx = np.linspace(
                        0,
                        len(flat) - 1,
                        50000,
                        dtype=np.int64
                    )

                    flat = flat[idx]

                sampled.append(
                    flat.astype(
                        np.float32,
                        copy=False
                    )
                )

            if not sampled:
                raise ValueError(
                    "No valid pixels"
                )

            all_pixels = np.concatenate(
                sampled
            )

            p1 = float(
                np.percentile(
                    all_pixels,
                    1
                )
            )

            p99 = float(
                np.percentile(
                    all_pixels,
                    99
                )
            )

            if (
                not np.isfinite(p1)
                or not np.isfinite(p99)
                or p99 <= p1
            ):
                raise ValueError(
                    "Invalid percentile range"
                )

            percentile_rows.append({
                "SeriesInstanceUID":
                    series_uid,
                "P1": p1,
                "P99": p99,
            })

        except Exception:
            bad_percentile_series += 1


percentiles_df = pd.DataFrame(
    percentile_rows
)

if (
    len(test_triplets_df) > 0
    and len(percentiles_df) > 0
):

    test_manifest = (
        test_triplets_df
        .merge(
            percentiles_df,
            on="SeriesInstanceUID",
            how="inner"
        )
    )

else:

    test_manifest = pd.DataFrame()


print(
    "Valid inference triplets:",
    len(test_manifest)
)

print(
    "Bad percentile series skipped:",
    bad_percentile_series
)

Computing P1/P99:   0%|          | 0/15 [00:00<?, ?it/s]

Valid inference triplets: 527
Bad percentile series skipped: 0


In [6]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

TARGET_SPACING = 0.5
IMAGE_SIZE = 224


def preprocess_slice(
    path,
    p1,
    p99,
    spacing_y,
    spacing_x
):

    arr = read_rescaled_pixels(path)

    arr = np.nan_to_num(
        arr,
        nan=p1,
        posinf=p99,
        neginf=p1
    )

    arr = np.clip(
        arr,
        p1,
        p99
    )

    denom = max(
        p99 - p1,
        1e-6
    )

    arr = (
        (arr - p1)
        / denom
    ).astype(np.float32)

    h, w = arr.shape

    new_h = max(
        1,
        int(round(
            h * spacing_y
            / TARGET_SPACING
        ))
    )

    new_w = max(
        1,
        int(round(
            w * spacing_x
            / TARGET_SPACING
        ))
    )

    x = torch.from_numpy(
        arr
    )[None, None]

    x = F.interpolate(
        x,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )[0, 0]

    h, w = x.shape

    if h > IMAGE_SIZE:
        top = (
            h - IMAGE_SIZE
        ) // 2
        x = x[
            top:top + IMAGE_SIZE,
            :
        ]

    if w > IMAGE_SIZE:
        left = (
            w - IMAGE_SIZE
        ) // 2
        x = x[
            :,
            left:left + IMAGE_SIZE
        ]

    h, w = x.shape

    pad_top = max(
        (IMAGE_SIZE - h) // 2,
        0
    )

    pad_bottom = max(
        IMAGE_SIZE - h - pad_top,
        0
    )

    pad_left = max(
        (IMAGE_SIZE - w) // 2,
        0
    )

    pad_right = max(
        IMAGE_SIZE - w - pad_left,
        0
    )

    x = F.pad(
        x,
        (
            pad_left,
            pad_right,
            pad_top,
            pad_bottom
        )
    )

    return x


class TestDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(
            drop=True
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        try:

            images = [
                preprocess_slice(
                    row[path_col],
                    float(row["P1"]),
                    float(row["P99"]),
                    float(row["PixelSpacingY"]),
                    float(row["PixelSpacingX"])
                )
                for path_col in [
                    "PreviousPath",
                    "CentrePath",
                    "NextPath"
                ]
            ]

            image = torch.stack(
                images,
                dim=0
            )

            valid = True

        except Exception:

            # Safe dummy image.
            # This triplet is ignored later.
            image = torch.zeros(
                3,
                IMAGE_SIZE,
                IMAGE_SIZE,
                dtype=torch.float32
            )

            valid = False

        return (
            image,
            str(row["StudyInstanceUID"]),
            valid
        )


test_dataset = TestDataset(
    test_manifest
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Dataset:", len(test_dataset))

Dataset: 527


In [7]:
all_probs = []
all_studies = []

model.eval()

with torch.no_grad():

    for (
        images,
        studies,
        valid
    ) in tqdm(
        test_loader,
        desc="Inference"
    ):

        valid = np.asarray(
            valid,
            dtype=bool
        )

        if not valid.any():
            continue

        images_valid = images[
            torch.from_numpy(valid)
        ].to(
            device,
            non_blocking=True
        )

        logits = model(
            images_valid
        )

        probs = torch.sigmoid(
            logits
        ).cpu().numpy()

        valid_studies = [
            studies[i]
            for i in range(len(studies))
            if valid[i]
        ]

        all_probs.append(
            probs
        )

        all_studies.extend(
            valid_studies
        )


if all_probs:

    all_probs = np.concatenate(
        all_probs,
        axis=0
    )

    pred_df = pd.DataFrame(
        all_probs,
        columns=TARGETS
    )

    pred_df.insert(
        0,
        "StudyInstanceUID",
        all_studies
    )

else:

    pred_df = pd.DataFrame(
        columns=[
            "StudyInstanceUID",
            *TARGETS
        ]
    )


print(
    "Valid predictions:",
    len(pred_df)
)

Inference:   0%|          | 0/9 [00:00<?, ?it/s]

Valid predictions: 527


In [8]:
# Start from a guaranteed-valid submission template
submission = sample_sub.copy()

# Put safe fallback probabilities everywhere first
for target in TARGETS:
    submission[target] = FALLBACK[target]


# Replace fallback with model predictions where available
if len(pred_df) > 0:

    study_pred = (
        pred_df
        .groupby(
            "StudyInstanceUID"
        )[TARGETS]
        .mean()
        .reset_index()
    )

    submission = (
        submission[
            ["StudyInstanceUID"]
        ]
        .merge(
            study_pred,
            on="StudyInstanceUID",
            how="left"
        )
    )

    # Fill studies with no valid MRI prediction
    for target in TARGETS:
        submission[target] = (
            submission[target]
            .fillna(
                FALLBACK[target]
            )
            .clip(0, 1)
        )


# Final defensive cleanup
for target in TARGETS:

    submission[target] = pd.to_numeric(
        submission[target],
        errors="coerce"
    )

    submission[target] = (
        submission[target]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(
            FALLBACK[target]
        )
        .clip(0, 1)
    )


# Preserve exact Kaggle column order
submission = submission[
    sample_sub.columns
]


print(
    "Submission shape:",
    submission.shape
)

print(
    "Expected:",
    sample_sub.shape
)

print(
    "Missing:",
    submission.isna().sum().sum()
)

print(
    "Duplicate IDs:",
    submission[
        "StudyInstanceUID"
    ].duplicated().sum()
)

print(
    "Studies receiving model predictions:",
    submission[
        "StudyInstanceUID"
    ].isin(
        pred_df[
            "StudyInstanceUID"
        ].unique()
        if len(pred_df)
        else []
    ).sum()
)

print(
    "Studies using fallback:",
    len(submission)
    -
    submission[
        "StudyInstanceUID"
    ].isin(
        pred_df[
            "StudyInstanceUID"
        ].unique()
        if len(pred_df)
        else []
    ).sum()
)


submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(
    "Saved /kaggle/working/submission.csv"
)

Submission shape: (3, 13)
Expected: (3, 13)
Missing: 0
Duplicate IDs: 0
Studies receiving model predictions: 3
Studies using fallback: 0
Saved /kaggle/working/submission.csv
